In [ ]:
pip install xmltodict lxml pydantic httpx python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

import httpx
import json
import requests as r
import pandas as pd
from io import BytesIO
from zipfile import ZipFile
from data.flow import xml_survey
from typing import List, Optional
from util.util import readxml, Cipher
from util.flow import get_token, get_headers, export_spreadsheet
from util.odk import odk
from core.dev import Dev

In [ ]:
FLOW_SERVICE_URL="https://fiji-dws.akvoflow.org/"
flow_api = "https://api-auth0.akvo.org/flow/orgs"
flow_instances = pd.read_csv("./data/flow-survey-amazon-aws.csv")
USERNAME = os.environ["MIS_AUTH0_USER"]
PASSWORD = os.environ["MIS_AUTH0_PWD"]
INSTANCE_NAME = 'fiji-dws'

In [ ]:
def get_folders(
    headers: dict,
    instance_name: str,
    id: Optional[int] = None,
):
    instance = flow_instances[flow_instances.instances.eq(instance_name)]
    if not instance.shape[0]:
        print("Instance Not Found")
        return
    url = f"{flow_api}/{instance_name}"
    folder_url = f"{url}/folders?parent_id=0"
    survey_url = f"{url}/surveys?folder_id=0"
    if id:
        folder_url = f"{url}/folders?parent_id={id}"
        survey_url = f"{url}/surveys?folder_id={id}"
    folders = r.get(folder_url, headers=headers)
    folders = folders.json().get("folders")
    surveys = r.get(survey_url, headers=headers)
    surveys = surveys.json().get("surveys")
    print({"folders": folders, "surveys": surveys})

In [ ]:
def get_survey_forms(
    headers: dict,
    instance_name: str,
    id: int
):
    url = f"{flow_api}/{instance_name}/surveys/{id}"
    req = r.get(url, headers=headers)
    data = req.json().get("forms")
    # [d.update({"surveyId": id}) for d in data]
    return data

In [ ]:
def download_sqlite_asset(
    cascade_list: List[str],
    ziploc: str,
) -> None:
    for cascade in cascade_list:
        cascade_file = cascade.split("/surveys/")[1]
        cascade_file = f"{ziploc}/{cascade_file}"
        cascade_file = cascade_file.replace(".zip", "")
        if not os.path.exists(cascade_file):
            try:
                zip_url = httpx.get(cascade)
                zip_url.raise_for_status()
                z = ZipFile(BytesIO(zip_url.content))
                # rename extracted file to include survey id
                z.extractall(ziploc)
            except httpx.HTTPError:
                pass
            # except httpx.HTTPError as exc:
            # raise HTTPException(
            #     status_code=exc.response.status_code,
            #     detail=f"Error while requesting {exc.request.url!r}.")

In [ ]:
def download_form(ziploc: str, alias: str, survey_id: int):
    instance = xml_survey(alias)
    dev = Dev()
    xml_path = f"{ziploc}/{survey_id}.xml"
    if dev.get_cached(xml_path):
        return readxml(xml_path=xml_path, alias=alias)
    try:
        zip_url = httpx.get(f"{instance}/{survey_id}.zip")
        zip_url.raise_for_status()
    except httpx.HTTPError:
        return False
    if not os.path.exists(ziploc):
        os.mkdir(ziploc)
    z = ZipFile(BytesIO(zip_url.content))
    z.extractall(ziploc)
    response = readxml(xml_path=xml_path, alias=alias)
    cascade_list = []
    # create csv to list question cascades
    cascade_csv_path = f"./output/flow_cascades/{survey_id}_cascades.csv"
    # loop through questions to find cascades with format columns:
    # survey_id, question_id, cascade_resource
    cascade_rows = []
    for qg in response["questionGroup"]:
        for q in qg.get("question", []):
            if q["type"] == "cascade":
                cascade_list.append(q["cascadeResource"])
                cascade_rows.append({
                    "survey_id": survey_id,
                    "question_id": q["id"],
                    "cascade_resource": q["cascadeResource"],
                })
    # Create DataFrame from accumulated list
    df = pd.DataFrame(cascade_rows) if cascade_rows else pd.DataFrame(columns=["survey_id", "question_id", "cascade_resource"])
    # if df not empty, save to csv
    if not df.empty:
        df.to_csv(cascade_csv_path, index=False)
    if len(cascade_list) > 0:
        cascade_list = [f"{instance}/{c}.zip" for c in cascade_list]
        sqlite_loc = "./output/flow_sqlite"
        download_sqlite_asset(cascade_list=cascade_list, ziploc=sqlite_loc)
    return response

In [ ]:
def get_form(id: str, save_json: bool = True):
    alias, survey_id = Cipher(id).decode()
    if alias is None:
        print("Alias is None!")
        return
    ziploc = f"./static/xml/{alias}"
    form_data = download_form(ziploc, alias, survey_id)
    if not form_data:
        print("Form not found")
        return
    # Save as JSON if requested
    if save_json:
        output_dir = "./output/flow_forms"
        os.makedirs(output_dir, exist_ok=True)
        
        form_name = form_data.get("name", "unknown")
        # Sanitize the name for filename (replace invalid characters)
        safe_name = "".join(c if c.isalnum() or c in ('-', '_', ' ') else '_' for c in form_name)
        safe_name = safe_name.replace(' ', '_')
        
        output_file = f"{output_dir}/{survey_id}_{safe_name}.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(form_data, f, indent=2, ensure_ascii=False)
        print(f"Form saved to: {output_file}")
    return form_data

In [ ]:
def file_log(dir_name: str):
    dir_name = dir_name.replace(
        f"./tmp/data_backup/{INSTANCE_NAME}.akvoflow.org", "")
    dir_name = dir_name.replace("//", "/")
    print(f"{dir_name}")

In [ ]:
def create_folder(dir: str, dir_name: str):
    if dir_name:
        dir += f"/{dir_name}/"
    else:
        dir += "/untitled/"
    if not os.path.exists(dir):
        os.makedirs(dir)
        file_log(f"CREATED : {dir}")
    else:
        file_log(f"EXISTS  : {dir}")
    return dir

In [ ]:
def flow_backup(headers: dict, survey_url: str, folder_url: str, dir: str):
    folders = r.get(folder_url, headers=headers)
    folders = folders.json().get("folders")
    surveys = r.get(survey_url, headers=headers)
    surveys = surveys.json().get("surveys")
    if not surveys:
        return
    for survey in surveys:
        dir_name = survey.get("name")
        form_url = survey.get("surveyUrl")
        survey_dir = create_folder(dir=dir, dir_name=dir_name)
        req = r.get(form_url, headers=headers)
        if req.status_code == 200:
            forms = req.json().get("forms")
            for form in forms:
                form_id = form.get("id")
                form_name = form.get("name")
                file_name = f"{form_id}-{form_name}"
                data_file = f"{survey_dir}/DATA-{file_name}.xlsx"
                if not os.path.exists(data_file):
                    try:
                        created = export_spreadsheet(
                            instance=INSTANCE_NAME,
                            survey_id=survey.get("id"),
                            form_id=form_id,
                            token=headers["refresh_token"],
                            custom_location=data_file)
                        if created:
                            file_log(f"CREATED : {data_file}")
                        else:
                            file_log(f"EMPTY   : {data_file}")
                    except Exception as e:
                        print(f"!!ERROR  : {str(e)}")
                else:
                    file_log(f"EXISTS  : {data_file}")
                form_file = f"{survey_dir}/FORM-{file_name}.xlsx"
                if not os.path.exists(form_file):
                    try:
                        ziploc = f"./static/xml/{INSTANCE_NAME}"
                        res = download_form(ziploc, INSTANCE_NAME, form_id)
                        if res:
                            odk(res, f"{survey_dir}/FORM-{file_name}.xlsx")
                            file_log(f"CREATED : {form_file}")
                        else:
                            file_log(f"!!ERROR : FILE: {form_file}")
                    except Exception as e:
                        print(f"!!ERROR  : {str(e)}")
                else:
                    file_log(f"EXISTS  : {form_file}")
    for folder in folders:
        dir_name = folder.get("name")
        new_dir = create_folder(dir=dir, dir_name=dir_name)
        survey_url = folder.get("surveysUrl")
        folder_url = folder.get("foldersUrl")
        flow_backup(
            headers=headers,
            survey_url=folder.get("surveysUrl"),
            folder_url=folder.get("foldersUrl"),
            dir=new_dir
        )

In [ ]:
def download_all_forms(form_ids: list = []):
    dev = Dev()
    forms = []
    for form_id in form_ids:
        form_chipper = dev.generate(
            alias=INSTANCE_NAME,
            fid=form_id
        )
        form_res = get_form(id=form_chipper, save_json=True)
        forms.append(form_res)
    return forms

In [ ]:
res = get_token(username=USERNAME, password=PASSWORD)
refresh_token = res.get("refresh_token")
headers = get_headers(refresh_token)

In [ ]:
def fetch_datapoint(url: str, headers: dict):
    response = r.get(url, headers=headers)
    status_code = response.status_code
    if status_code == 200:
        response = response.json()
        return response
    return None

In [ ]:
def create_data_csv(data: dict, form_id: int, form_name: str):
    """
    Create CSV file from form instances data in transposed format
    Each row represents a datapoint, each column represents a question
    
    Args:
        data: Dictionary containing formInstances
        form_id: ID of the form
        form_name: Name of the form
    """
    try:
        # Get form instances
        formInstances = data.get("formInstances", [])
        if not formInstances:
            print(f"No form instances found for form_id: {form_id}")
            return 0
        
        # Sort by creation date
        formInstances.sort(key=lambda fi: fi.get("createdAt", ""), reverse=False)
        
        # Prepare data rows - each row is ONE datapoint with all questions as columns
        rows = []
        
        for fi in formInstances:
            try:
                datapoint_id = fi.get("dataPointId")
                responses = fi.get("responses", {})
                
                # Create a single row for this datapoint with all questions
                row = {"datapoint_id": datapoint_id}
                
                # Process each question response - flatten all into one row
                for key, value in responses.items():
                    if not isinstance(value, list):
                        continue
                    
                    for val in value:
                        if not isinstance(val, dict):
                            continue
                        
                        # Each question_id becomes a column
                        for question_id, answer in val.items():
                            # Convert complex types to JSON string, keep simple types as-is
                            if isinstance(answer, (dict, list)):
                                row[str(question_id)] = json.dumps(answer, ensure_ascii=False)
                            elif answer is not None:
                                row[str(question_id)] = answer
                            else:
                                row[str(question_id)] = ""
                
                # Only add row if it has data beyond datapoint_id
                if len(row) > 1:
                    rows.append(row)
                
            except Exception as e:
                print(f"Error processing form instance for datapoint {datapoint_id}: {str(e)}")
                continue
        
        if not rows:
            print(f"No rows generated for form_id: {form_id}")
            return 0
        
        # Create DataFrame - each row is one datapoint, columns are questions
        df = pd.DataFrame(rows)
        
        # Add row number column at the beginning
        df.insert(0, 'number', range(1, len(df) + 1))
        
        # Prepare output directory and file
        output_dir = "./output/flow_data"
        os.makedirs(output_dir, exist_ok=True)
        
        # Sanitize form name for filename
        safe_name = "".join(c if c.isalnum() or c in ('-', '_', ' ') else '_' for c in form_name)
        safe_name = safe_name.replace(' ', '_')
        
        csv_file = f"{output_dir}/{form_id}_{safe_name}.csv"
        
        # Append to existing file or create new one
        if os.path.exists(csv_file):
            # Read existing file and append
            existing_df = pd.read_csv(csv_file)
            # Update row numbers to continue from last number
            df["number"] = df["number"] + existing_df["number"].max()
            # Combine dataframes, filling missing columns with empty strings
            combined_df = pd.concat([existing_df, df], ignore_index=True, sort=False)
            combined_df.fillna("", inplace=True)
            combined_df.to_csv(csv_file, index=False, encoding='utf-8')
        else:
            df.fillna("", inplace=True)
            df.to_csv(csv_file, index=False, encoding='utf-8')
        
        print(f"Saved {len(df)} rows (datapoints) to {csv_file}")
        
        # Write to log file
        log_file = "./output/flow_data/download_log.csv"
        log_exists = os.path.exists(log_file)
        
        # Get current total rows from the saved file
        final_df = pd.read_csv(csv_file)
        total_rows = len(final_df)
        
        log_entry = pd.DataFrame([{
            "timestamp": pd.Timestamp.now().isoformat(),
            "form_id": form_id,
            "form_name": form_name,
            "rows_added": len(df),
            "total_rows": total_rows
        }])
        
        if log_exists:
            log_entry.to_csv(log_file, mode='a', header=False, index=False, encoding='utf-8')
        else:
            log_entry.to_csv(log_file, index=False, encoding='utf-8')
        
        return len(df)
        
    except Exception as e:
        print(f"Error in create_data_csv for form_id {form_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0

In [ ]:
def download_all_data(
    headers: dict, forms: list = []
):
    for form in forms:
        form_id = form.get("surveyId")
        survey_id = form.get("surveyGroupId")
        form_name = form.get("name", "unknown")
        
        print(f"\n{'='*60}")
        print(f"Downloading data for: {form_name} (Form ID: {form_id})")
        print(f"{'='*60}")
        
        url = f"{flow_api}/{INSTANCE_NAME}/form_instances?survey_id={survey_id}&form_id={form_id}"
        data = fetch_datapoint(url=url, headers=headers)
        if data is None:
            print(f"No data found for form_id: {form_id}, survey_id: {survey_id}")
            continue
        
        # Process the first page
        total_rows = create_data_csv(data=data, form_id=form_id, form_name=form_name)
        
        # Process remaining pages
        nextPageUrl = data.get("nextPageUrl")
        page_count = 1
        while nextPageUrl:
            if nextPageUrl is None or str(nextPageUrl).strip() == "":
                break
            page_count += 1
            print(f"Fetching page {page_count}...")
            data = fetch_datapoint(url=nextPageUrl, headers=headers)
            if data is None:
                break
            rows_added = create_data_csv(data=data, form_id=form_id, form_name=form_name)
            total_rows += rows_added
            nextPageUrl = data.get("nextPageUrl")
        
        print(f"\nCompleted: {form_name}")
        print(f"Total pages processed: {page_count}")
        print(f"Total rows downloaded: {total_rows}")


In [ ]:
flow_ids = [
    8520967, # WTP
    17260923, # WWTP
    27040920, # SPS (Pump Station)
    1520924, # EPS Water quality
    5530933, # EPS Project Construction
    2490944, # RWS
]
forms = download_all_forms(form_ids=flow_ids)
download_all_data(headers=headers, forms=forms)
    